# 🥇 Train Custom YOLO11-Pose to Detect Grasp Points

สมุดโค้ดนี้จะรับดาต้าเซ็ต Kaggle ของท่านที่มีภาพตะกร้ายา (`train`) และ Label JSON (`train-corrected`) แล้วตีตื้นแปลงฟอร์แมตเอาไป **Train Custom YOLO-Pose Model** เพื่อให้หาสี่เหลี่ยมคลุมขวด และพิกัดจุดหยิบ (Centroid Keypoint) จากตะกร้ามุมลงได้โดยตรง!

## Before use this code, push train-label form พรี่หมอ to colab (path /content)

## 1. Setup Kaggle Data & Install YOLO11

In [ ]:
!pip install -q kaggle ultralytics

import os
import json
import shutil
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from ultralytics import YOLO

# ⚠️ กรุณาตั้งค่า API ของบัญชี Kaggle อีกครั้ง หากต้องการโหลด Data อัตโนมัติ
os.environ['KAGGLE_API_TOKEN'] = '<api-key>'

!kaggle competitions download -c egco-696-project-2026 -p /content/
!unzip -q -o /content/egco-696-project-2026.zip -d /content/dataset/

### 📤 อัปโหลดโฟลเดอร์พิกัด Label (`train-corrected.zip`)
นำไฟล์ `train-corrected.zip` ไปวางในไฟล์ระบบของ Colab ที่หน้าแรก (`/content/`) ก่อนรันโค้ดบรรทัดด้านล่างนี้นะครับ เพื่อให้มันแตกไฟล์รอไว้ให้

In [ ]:
!unzip -q -o /content/train-corrected.zip -d /content/dataset/
print('แตกไฟล์ลงโฟลเดอร์ dataset เสร็จสิ้น! ✅')

## 2. จัดระเบียบโฟลเดอร์สำหรับ Pose Format
YOLO-Pose ต้องการการจัดเรียงรูปภาพและไฟล์ `.txt` แยกกันตาม `train` และ `val`

In [ ]:
DATASET_DIR = '/content/dataset'
IMG_DIR = f'{DATASET_DIR}/Basket_medicine/train'
JSON_DIR = f'{DATASET_DIR}/train-corrected'  # ตรงพาทแล้วครับ!

POSE_DIR = '/content/dataset/pose'
os.makedirs(f'{POSE_DIR}/images/train', exist_ok=True)
os.makedirs(f'{POSE_DIR}/images/val', exist_ok=True)
os.makedirs(f'{POSE_DIR}/labels/train', exist_ok=True)
os.makedirs(f'{POSE_DIR}/labels/val', exist_ok=True)

## 3. ดึงข้อมูลจาก JSON (COCO-like) มาแปลงเป็น `.txt` YOLO Format

In [ ]:
# ลิสต์ไฟล์ JSON ทั้งหมดในโฟลเดอร์
json_files = [f for f in os.listdir(JSON_DIR) if f.endswith('.json')]
print(f"Found {len(json_files)} JSON label files.")

# แบ่ง 80% สำหรับหน้า Train และ 20% สำหรับตรวจเช็ค (Validation)
random.seed(42)
random.shuffle(json_files)
split_idx = int(len(json_files) * 0.8)
train_jsons = json_files[:split_idx]
val_jsons = json_files[split_idx:]

def convert_and_copy(files, split_name):
    for j_file in tqdm(files, desc=f'Processing {split_name}'):
        j_path = os.path.join(JSON_DIR, j_file)
        
        try:
            with open(j_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
            # บางไฟล์อาจไม่มีรูป
            if 'images' not in data or len(data['images']) == 0:
                continue    
                
            img_info = data['images'][0]
            img_name = img_info['file_name']
            img_w = img_info['width']
            img_h = img_info['height']
            
            src_img = os.path.join(IMG_DIR, img_name)
            if not os.path.exists(src_img):
                continue
            
            labels_text = []
            for ann in data.get('annotations', []):
                # ข้อมูลเดิม
                x, y, w, h = ann['bbox']
                cx, cy = ann['centroid']
                
                # 1. กล่อง Bounding Box หาร Normalize ขนาด
                cx_box = (x + (w / 2.0)) / img_w
                cy_box = (y + (h / 2.0)) / img_h
                nw = w / img_w
                nh = h / img_h
                
                # 2. Keypoint หารพิกัด
                nc_x = cx / img_w
                nc_y = cy / img_h
                # 2 คือเห็นชัดเจน (Labeled and visible)
                visibility = 2 
                
                labels_text.append(f"0 {cx_box:.6f} {cy_box:.6f} {nw:.6f} {nh:.6f} {nc_x:.6f} {nc_y:.6f} {visibility}")
                
            # ย้ายภาพและเขียนไฟล์ txt ไปตั้งพาทใหม่
            shutil.copy(src_img, f'{POSE_DIR}/images/{split_name}/{img_name}')
            txt_name = img_name.replace('.png', '.txt').replace('.jpg', '.txt')
            with open(f'{POSE_DIR}/labels/{split_name}/{txt_name}', 'w') as tf:
                tf.write('\n'.join(labels_text))
                
        except Exception as e:
            print(f"Skip file {j_file} due to: {e}")

convert_and_copy(train_jsons, 'train')
convert_and_copy(val_jsons, 'val')
print("Conversion Completed! ✅")

## 4. สร้าง Config ยิงเข้าเตาเผา (Training)

In [ ]:
# เขียนไฟล์ dataset.yaml ให้อธิบายเส้นทางที่จะไปเทรนให้ YOLO ทราบ
yaml_content = f"""path: {POSE_DIR}
train: images/train
val: images/val

# 1 Point, 3 ค่าต่อพิกัด (x, y, visibility)
kpt_shape: [1, 3] 

names:
  0: medicine
"""
with open(f'{POSE_DIR}/bottle-pose.yaml', 'w') as f:
    f.write(yaml_content)
    
print("YAML Config generated!")

In [ ]:
# 🔥 สั่ง Train YOLO-Pose รีดประสิทธิภาพ VRAM แบบสุดหลอด! (รุ่น L, ภาพ 1280x1280)
!yolo task=pose mode=train data={POSE_DIR}/bottle-pose.yaml model=yolo11l-pose.pt epochs=60 imgsz=1280 batch=4 name=medicine_grasp_v4 patience=20 mosaic=0.0 mixup=0.0 hsv_v=0.7 hsv_s=0.7 erasing=0.6 fliplr=0.5 flipud=0.5 degrees=0.0


## 5. ทำนายข้อสอบหาจุดบนไฟล์จากโมเดลที่ตีบวกใหม่! (Test Set Prediction)

In [ ]:
TEST_DIR = f'{DATASET_DIR}/Basket_medicine/test'
SUBMISSION_PATH = '/content/submission_hybrid_sam.csv'
SAMPLE_SUB_PATH = f'{DATASET_DIR}/Basket_medicine/sample-submission.csv'

from ultralytics import SAM
import numpy as np

# 1. โมเดลสายตาเหยี่ยว (YOLO-Pose รุ่น L) เอาไว้ตีกรอบ Bounding Box ที่แม่นยำที่สุด
trained_model = YOLO('/content/runs/pose/medicine_grasp_v4/weights/best.pt')

# 2. โมเดลสายลับ (FastSAM) เอาไว้ตัดฉากหลังและวาด Mask ระบายสีขวดแบบ Zero-shot
sam_model = SAM('sam2.1l.pt') # อัปเกรดเป็น SAM 2.0 (ล้ำหน้าที่สุดของโลกตอนนี้)\n

sample_df = pd.read_csv(SAMPLE_SUB_PATH)
results_list = []
print(f"Predicting on {len(sample_df)} test images with Hybrid Pipeline (YOLO Box -> SAM Mask -> Centroid)...")

for _, s_row in tqdm(sample_df.iterrows(), total=len(sample_df), desc='Scoring'):
    jpg_id = s_row['Id']
    img_name = jpg_id.replace('.jpg', '.png')
    img_path = os.path.join(TEST_DIR, img_name)
    
    img = cv2.imread(img_path)
    row_result = {'Id': jpg_id}
    grasp_points = []
    
    if img is not None:
        h, w = img.shape[:2]
        
        # Phase 1: สั่ง YOLO ให้หากล่อง Bounding Box ที่มั่นใจระดับสูง (ลด False Positives)
        preds = trained_model(img, conf=0.75, imgsz=1280, augment=True, verbose=False)
        
        for p in preds:
            if len(p.boxes.xyxy) > 0:
                # ดึงพิกัดกล่องทั้งหมดออกมา
                bboxes = p.boxes.xyxy.cpu().numpy()
                
                # Phase 2: เอาพิกัดกล่องไปสั่งให้ SAM วาด Mask แผ่นภาพของขวดให้ตัดกับขอบพอดี
                sam_results = sam_model(img, bboxes=bboxes, verbose=False)
                
                for sam_res in sam_results:
                    if sam_res.masks is not None:
                        # ดึงแผ่น Mask ออกมาทีละขวด
                        for mask in sam_res.masks.data:
                            # แปลงแผ่น Mask เป็นอาเรย์คณิตศาสตร์ ขาวดำ (0 กับ 1)
                            mask_np = mask.cpu().numpy().astype(np.uint8)
                            
                            # Phase 3: ใช้ OpenCV คำนวณสมดุลโมเมนต์ (Center of Mass)
                            M = cv2.moments(mask_np)
                            if M["m00"] != 0:
                                cx = int(M["m10"] / M["m00"])
                                cy = int(M["m01"] / M["m00"])
                                
                                # แปลงเป็นระยะหารความกว้างยาว (อิงกติกา Normalized) แล้วเก็บคำตอบ
                                grasp_points.append((cx / w, cy / h))
                                

    # ---- NEW: SPATIAL NMS (ต้านจุดซ้ำซ้อนตีกินคะแนน Extra Penalty) ----
    filtered_points = []
    for px, py in grasp_points:
        is_duplicate = False
        for fx, fy in filtered_points:
            # คำนวณระยะห่างจุด Euclidean Distance (Normalized 0-1)
            dist = np.sqrt((px - fx)**2 + (py - fy)**2)
            if dist < 0.04:  # ถ้าระยะห่างน้อยกว่า 4% ของพื้นที่ภาพ (ประมาณ 50 pixels) ถือว่าเป็นขวดเดียวกัน!
                is_duplicate = True
                break
        if not is_duplicate:
            filtered_points.append((px, py))
    
    grasp_points = filtered_points
    # -----------------------------------------------------------------
    # หยิบ 7 คำตอบที่ดีที่สุดมาส่ง
    grasp_points = grasp_points[:7]
    
    for i in range(7):
        if i < len(grasp_points):
            row_result[f'x{i+1}'] = grasp_points[i][0]
            row_result[f'y{i+1}'] = grasp_points[i][1]
        else:
            row_result[f'x{i+1}'] = -1.0
            row_result[f'y{i+1}'] = -1.0
            
    results_list.append(row_result)

### 🔎 ส่องดูผลงานพรีวิวก่อนเซฟ (Visual Review)

In [ ]:
# เพื่อความมั่นใจ ขอดูสัก 5 รูปแรกหน่อยว่ามันจิ้มจุดตรงขวดไหมจารย์!
num_samples_to_show = 5
print(f"\n--- กำลังแสดงภาพตัวอย่างที่ทำนายได้ {num_samples_to_show} รูป ---")

for i, row in enumerate(results_list[:num_samples_to_show]):
    img_id = row['Id'].replace('.jpg', '.png')
    img_path = os.path.join(TEST_DIR, img_id)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(f"Grasp Predictions: {img_id}")
    
    for j in range(1, 8):
        x_norm = row[f'x{j}']
        y_norm = row[f'y{j}']
        if x_norm != -1.0 and y_norm != -1.0:
            px, py = int(x_norm * w), int(y_norm * h)
            # วาดกากบาทสีแดงตัวใหญ่กากับบนขวดเลย
            plt.scatter(px, py, c='red', s=150, marker='x', linewidths=3)
            
    plt.axis('off')
    plt.show()

### 💾 บันทึกไฟล์ไปส่ง Submission

In [ ]:
# Save เมื่อพึงพอใจแล้ว!
df_submission = pd.DataFrame(results_list)
df_submission.to_csv(SUBMISSION_PATH, index=False)
print(f"✅ Custom Submission saved to {SUBMISSION_PATH}")
df_submission.head()